<a href="https://colab.research.google.com/github/Shashith240/Statistical-Learning-e22240/blob/main/Assignment_7d.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


## Q. Bayesian Estimations for Structural Health Monitoring via Bounded Grid Updates

In structural health monitoring (SHM), sensors placed along a bridge structure continuously record binary stress/strain thresholds over independent measurement cycles $k$. Let $\Theta = \theta$ be the latent structural integrity parameter representing the true health index of a critical joint, restricted to a bounded domain $\theta \in [-5, 5]$.

Let $Y_k$ denote the binary sensor threshold trigger at cycle $k$:


$$Y_k = \begin{cases} 1, & \text{if a stress deflection anomaly is recorded} \\ 0, & \text{if structural baseline behavior is nominal} \end{cases}$$

The conditional anomaly likelihood follows a specialized two-parameter logistic failure distribution. Given the latent structural condition $\theta$, the probability of registering an anomaly for a sensor characterized by operational baseline sensitivity $b_k$ (difficulty/threshold parameter) and structural fault discrimination intensity $a_k > 0$ is defined by:


$$P(Y_k = 1 \mid \Theta = \theta) = p_k(\theta) = \frac{1}{1 + e^{a_k(\theta - b_k)}}$$


*Note: The negative coefficient changes direction here because lower latent integrity $\theta$ yields higher probabilities of experiencing a deflection anomaly.*

The prior health belief is initialized using a standard normal distribution over the bounded spatial domain:


$$f_{\Theta}^{(0)}(\theta) \propto \exp\left(-\frac{\theta^2}{2}\right), \quad \theta \in [-5, 5]$$

---

## 1. Visualizing Sensor Mechanics

The baseline response function maps out how changing underlying integrity levels trigger stress warnings.

* **Interpretation of $b_k$ Shifts:** The parameter $b_k$ represents the exact point of structural inflection where the probability of generating a stress anomaly is exactly $50\%$ ($p_k(\theta) = 0.5$). Moving $b_k$ shifts the curve horizontally along the $\theta$-axis.
* A large positive value of $b_k$ translates the curve to the right, meaning the sensor acts as a highly sensitive safety monitor that triggers anomalies even when structural health is relatively high.
* A low or negative value of $b_k$ translates the curve to the left, acting as a critical point failure alarm that remains quiet until structural health has degraded significantly.



---

## 2. Sequential Likelihood Structure

For an isolated cycle $k$, the likelihood contribution $L(y_k \mid \theta)$ of a single observed binary sensor response $y_k \in \{0, 1\}$ given the joint health index $\theta$ is represented as a Bernoulli variant:


$$L(y_k \mid \theta) = [p_k(\theta)]^{y_k} [1 - p_k(\theta)]^{1 - y_k}$$

Given the assumption of conditional independence of sensor logs over independent structural cycles, the joint likelihood function for the running history vector $y^{(k)} = (y_1, y_2, \dots, y_k)$ is the product of individual steps:


$$\mathcal{L}(y^{(k)} \mid \theta) = \prod_{i=1}^{k} [p_i(\theta)]^{y_i} [1 - p_i(\theta)]^{1 - y_i}$$

---

## 3. Mathematical Formulation of the Running Update

Under a recursive Bayesian tracking framework, the posterior distribution from monitoring cycle $k-1$ acts as the formal prior distribution for cycle $k$. Applying Bayes' theorem sequentially yields:


$$f_{\Theta \mid Y^{(k)}}(\theta \mid y^{(k)}) = \frac{L(y_k \mid \theta) \cdot f_{\Theta \mid Y^{(k-1)}}(\theta \mid y^{(k-1)})}{\int_{-5}^{5} L(y_k \mid s) \cdot f_{\Theta \mid Y^{(k-1)}}(s \mid y^{(k-1)}) \, ds}$$

Expressed up to a proportionality constant to focus purely on the kernel operations:


$$f_{\Theta \mid Y^{(k)}}(\theta \mid y^{(k)}) \propto \left[ [p_k(\theta)]^{y_k} [1 - p_k(\theta)]^{1 - y_k} \right] \cdot f_{\Theta \mid Y^{(k-1)}}(\theta \mid y^{(k-1)})$$

---

## 4. Dynamic Shifting Mechanics

When an anomaly is observed ($y_k = 1$) at a sensor with a high baseline sensitivity threshold (large positive $b_k$), it provides significant evidence of structural degradation.

Mathematically, the likelihood curve $[p_k(\theta)]^1$ dominates the regions where $\theta < b_k$, featuring a high profile on the lower-health side of the axis. Multiplying the prior density by this downward-sloping likelihood curve skews the distribution leftward. This pushes the new mode—the Maximum A Posteriori (MAP) estimate—and the posterior mean down toward a lower integrity value relative to cycle $k-1$.

---

## 5. Tracking Certainty and Sharpness

The discrimination parameter $a_k$ determines the gradient slope of the sensor response function.

* **Large $a_k$ (High Discrimination):** The likelihood function approaches a sharp step function. When updating, this introduces a localized, abrupt change in the posterior shape. It rapidly truncates improbable values of $\theta$, reducing the posterior variance and causing the distribution's peak to become significantly sharper.
* **Small $a_k$ (Low Discrimination/High Noise):** The curve updates slowly and remains flat across a wide span of $\theta$. This provides little information, preserving the variance of the previous step and leaving the sharpness of the distribution largely unchanged.

---

## 6. Numerical Implementation of a Bounded Grid

Because the logistic response model does not possess a conjugate prior, analytical tracking is closed. Instead, we approximate the density function numerically across a fixed mesh grid:

1. **Grid Initialization:** Define a linear vector $\vec{\theta}$ composed of $M$ equally spaced nodes spanning from $-5$ to $+5$, with a discrete step width $\Delta \theta = \frac{10}{M-1}$.
2. **Base Density:** Initialize an array representing the prior: $\vec{f}^{(0)} = \exp(-\vec{\theta}^2 / 2)$.
3. **Sequential Processing Matrix:** For each arriving sensor check $y_k$:
* Compute the vector of response probabilities across the mesh grid: $\vec{p}_k = (1 + \exp(a_k(\vec{\theta} - b_k)))^{-1}$.
* Compute the discrete likelihood array: $\vec{L}_k = (\vec{p}_k)^{y_k} \odot (1 - \vec{p}_k)^{1 - y_k}$.
* Perform an element-wise product update: $\vec{f}^{(k)}_{\text{unnorm}} = \vec{f}^{(k-1)} \odot \vec{L}_k$.


4. **Sequential Normalization:** Approximate the continuous denominator integral via the trapezoidal integration rule:

$$I = \text{np.trapezoid}(\vec{f}^{(k)}_{\text{unnorm}}, \vec{\theta})$$



Normalize the grid array to maintain a valid probability distribution: $\vec{f}^{(k)} = \vec{f}^{(k)}_{\text{unnorm}} / I$.

---

## 7. Numerical Simulation Script (Python)

```python
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

# Set random seed for structural simulation reproducibility
np.random.seed(2026)

# 1. Setup Simulation Parameters
theta_true = 0.75
n_cycles = 20
theta_grid = np.linspace(-5, 5, 1000)  # Discrete bounded grid mesh
delta_theta = theta_grid[1] - theta_grid[0]

# Generate sensor tracking profiles across timeline
a_params = np.random.uniform(0.5, 2.0, size=n_cycles)
b_params = np.random.normal(0, 1, size=n_cycles)

# 2. Initialize Tracking Arrays
running_bayes = [0.0]
running_map = [0.0]
cycles = list(range(n_cycles + 1))

# Initialize prior bounded density array: N(0, 1) truncated to grid
current_posterior = stats.norm.pdf(theta_grid, 0, 1)
current_posterior /= np.trapezoid(current_posterior, theta_grid)

# Define inverse link function for SHM
def p_k(theta, a, b):
    return 1.0 / (1.0 + np.exp(a * (theta - b)))

# 3. Running Update Loop
for k in range(n_cycles):
    a_k = a_params[k]
    b_k = b_params[k]
    
    # Calculate operational true warning probability
    prob_true = p_k(theta_true, a_k, b_k)
    # Stochastically simulate sensor outcome
    y_k = 1 if np.random.uniform(0, 1) < prob_true else 0
    
    # Grid computation of incoming structural likelihood
    prob_grid = p_k(theta_grid, a_k, b_k)
    likelihood = (prob_grid ** y_k) * ((1.0 - prob_grid) ** (1 - y_k))
    
    # Sequential Bayesian update execution
    current_posterior = current_posterior * likelihood
    normalization_constant = np.trapezoid(current_posterior, theta_grid)
    current_posterior /= normalization_constant
    
    # Extract numerical point estimators
    theta_bayes_k = np.trapezoid(theta_grid * current_posterior, theta_grid)
    theta_map_k = theta_grid[np.argmax(current_posterior)]
    
    running_bayes.append(theta_bayes_k)
    running_map.append(theta_map_k)

# 4. Interactive Plotly Visualization
fig = go.Figure()

# Add True Structural Reference Line
fig.add_hline(
    y=theta_true,
    line_dash="dash",
    line_color="red",
    line_width=2,
    annotation_text=f"True Structural Health Status (θ = {theta_true})",
    annotation_position="top right"
)

# Add Posterior Mean Trace
fig.add_trace(go.Scatter(
    x=cycles, y=running_bayes,
    mode='lines+markers',
    name='Posterior Mean (θ̂_Bayes)',
    line=dict(color='blue', width=2.5),
    marker=dict(size=6)
))

# Add MAP Trace
fig.add_trace(go.Scatter(
    x=cycles, y=running_map,
    mode='lines+markers',
    name='MAP Estimate (θ̂_MAP)',
    line=dict(color='green', width=2),
    marker=dict(size=6, symbol='square')
))

fig.update_layout(
    title={
        'text': "Sequential Structural Integrity Estimation over Inspection Cycles",
        'y': 0.93, 'x': 0.5, 'xanchor': 'center', 'yanchor': 'top'
    },
    xaxis_title="Monitoring Phase / Sensor Evaluation Cycle (k)",
    yaxis_title="Estimated Structural Integrity (θ̂)",
    xaxis=dict(tickmode='linear', tick0=0, dtick=2),
    yaxis=dict(range=[-1.5, 2.5]),
    template="plotly_white",
    hovermode="x unified"
)

fig.show()

```

### Analysis & Convergence Interpretation

As the inspection timeline $k \to 20$ advances, the distance between both estimators ($\hat{\theta}_{\text{Bayes}}^{(k)}$, $\hat{\theta}_{\text{MAP}}^{(k)}$) and the true structural index ($\theta_{\text{true}} = 0.75$) steadily decreases. This narrowing error tracking window indicates that the system is successfully aggregating sensory data.

As more measurements accumulate, the likelihood function dominates the initial normal prior. The variance of the underlying posterior array shrinks toward zero, reflecting growing confidence in the structural integrity estimate.